In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/bank-full.csv", sep=";")
df = df.drop(columns=["duration"])

print(df.shape)
print(df.columns.tolist())

(45211, 16)
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


In [2]:
y = (df["y"] == "yes").astype(int)
X = df.drop(columns=["y"])

X = pd.get_dummies(X, drop_first=True)

print(X.shape)
print(X.columns.tolist()[:12])

(45211, 41)
['age', 'balance', 'day', 'campaign', 'pdays', 'previous', 'job_blue-collar', 'job_entrepreneur', 'job_housemaid', 'job_management', 'job_retired', 'job_self-employed']


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Yes-rate in train:", round(y_train.mean(), 4))
print("Yes-rate in test:", round(y_test.mean(), 4))

Train: (36168, 41)
Test: (9043, 41)
Yes-rate in train: 0.117
Yes-rate in test: 0.117


In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape, X_test_scaled.shape)
print(round(X_train_scaled[:, 0].mean(), 3), round(X_train_scaled[:, 0].std(), 3))

(36168, 41) (9043, 41)
-0.0 1.0


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("Train accuracy:", round(model.score(X_train_scaled, y_train), 4))
print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Always-say-no accuracy:", round(1 - y_test.mean(), 4))
print("Predicted yes:", y_pred.sum(), "out of", len(y_pred))

Train accuracy: 0.8923
Test accuracy: 0.8935
Always-say-no accuracy: 0.883
Predicted yes: 285 out of 9043


In [6]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["no", "yes"]))

[[7890   95]
 [ 868  190]]
              precision    recall  f1-score   support

          no       0.90      0.99      0.94      7985
         yes       0.67      0.18      0.28      1058

    accuracy                           0.89      9043
   macro avg       0.78      0.58      0.61      9043
weighted avg       0.87      0.89      0.87      9043



In [7]:
model_bal = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
model_bal.fit(X_train_scaled, y_train)

y_pred_bal = model_bal.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred_bal))
print(classification_report(y_test, y_pred_bal, target_names=["no", "yes"]))

[[6175 1810]
 [ 399  659]]
              precision    recall  f1-score   support

          no       0.94      0.77      0.85      7985
         yes       0.27      0.62      0.37      1058

    accuracy                           0.76      9043
   macro avg       0.60      0.70      0.61      9043
weighted avg       0.86      0.76      0.79      9043

